# From laptop to NERSC: bringing your scientific notebook with you

Package the software for an ATLAS Open Data analysis, then run it through NERSC JupyterHub using a container-backed kernel.

**By the end, you’ll be able to:**

- Create and build a container image.
- Configure and select a Jupyter kernel that uses that container at NERSC.

**Note:** All demonstrations take place on Perlmutter. No container software is needed on your laptop.

## Table of Contents

- [1. Introduction](#1---Introduction)
- [2. ATLAS - Find the Z Boson analysis](#2---ATLAS---Find-the-Z-Boson-analysis)
- [3. Containers](#3---Containers)
- [4. Containers on NERSC JupyterHub](#4---Containers-on-NERSC-JupyterHub)
- [5. Running the Analysis](#5---Running-the-Analysis)
- [6. Summary and Further Exploration](#6---Summary-and-Further-Exploration)

<img src="images/tutorial-workflow.svg" alt="All steps take place at NERSC: start with the ATLAS notebook and software requirements (Section 2), package the software (Section 3), tell Jupyter to use it (Section 4), and run the analysis (Section 5)." width="1120" style="max-width: 100%; height: auto;">

We keep the notebook and analysis code, package the required software, and configure Jupyter to run Python inside that container.

## 1 - Introduction

### Why move your analysis to NERSC?

Your notebook may work well on your laptop, but you may want to:

- **Work where the data reside:** access datasets at NERSC, including those shared by your collaboration.
- **Use more memory:** run an analysis whose working data exceed your laptop’s RAM.
- **Access more computing capacity:** process many files or run multiple analyses in parallel.
- **Use specialized hardware:** access GPUs unavailable on your laptop.

### The notebook is only part of the analysis

A notebook needs Python, packages, and their supporting libraries. Copying the notebook doesn’t recreate that environment: missing packages or incompatible versions can stop a working analysis.

We’ll package the environment in a container and connect it to Jupyter at NERSC.

## 2 - ATLAS - Find the Z Boson analysis

ATLAS is a particle physics experiment at CERN’s Large Hadron Collider. Proton collisions produce particles whose energies and momenta are measured by the detector to study fundamental physics.

We’ll use [ATLAS Open Data](https://opendata.atlas.cern/) to reconstruct the Z boson mass from the two muons produced in its decay.

<img src="images/Zee_feynman.png" alt="Feynman diagram showing a Z boson decaying into a positron and an electron." width="600">

*The diagram illustrates a Z boson decaying into an electron–positron pair. Our analysis uses the corresponding decay into a muon–antimuon pair.*

Figure: [ATLAS Open Data, Find the Z notebook collection](https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/8b7ee0050ae0e34d072fb017da662be0c32ab97c/13-TeV-examples/uproot_python/images/feynman_diagrams/Zee_feynman.png), reproduced unchanged. See [license and provenance](UPSTREAM.md).

### What does the analysis do?

1. Read the measured energies and momenta of two muons.
2. Calculate their **invariant mass**—the mass reconstructed from their combined energy and momentum.
3. Plot the distribution and look for the Z boson peak.

The code is adapted from the [ATLAS Open Data *Find the Z* notebook](https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/8b7ee0050ae0e34d072fb017da662be0c32ab97c/13-TeV-examples/uproot_python/Find_the_Z.ipynb) into [Python functions](utils/find_the_z.py) for this tutorial. Converting a notebook to a script is not required to use containers.

## 3 - Containers

### What is a container?

A **container** runs an application in an isolated process environment. Our analysis uses the container's Python, packages, and system libraries while sharing the host's Linux kernel, rather than booting its own operating system.

A **container image** is the packaged software environment we build and store. Starting a process from that image creates a running container. In this tutorial, the image supplies the dependencies; the analysis code stays in our repository, made accessible inside the container through a mount.

Container tooling such as **podman-hpc** starts and manages the container; the host's Linux kernel provides the isolation. This operating-system kernel is different from the **Jupyter kernel** that runs our Python code in Section 4.

<img src="images/container-environment.svg" alt="An isolated container runs analysis code from the mounted repository using Python, packages, and system libraries supplied by the image. Beneath it are podman-hpc, the shared host Linux kernel, and host hardware." width="720">

*The image supplies the software environment; the running container shares the host's Linux kernel. [More about containers (Docker)](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/).*

### Why use a container?

**A container image packages our installed analysis dependencies together with supporting system libraries and tools.** We can share and reuse that image on compatible systems instead of installing the environment separately on each one.

Conda and containers play complementary roles here: Micromamba installs the packages; the container image packages them together with the supporting operating-system files and tools.

Packaging this environment lets us:

- **Share the same image with collaborators**, without each person recreating the environment.
- **Reuse it for other workflows**, such as running the analysis in a batch job or on another compatible system.

In this tutorial, we use that environment through a container-backed Jupyter kernel at NERSC.

<details>
<summary>Rebuilding and compatibility</summary>

An image preserves its software, but rebuilding can change unpinned dependencies or the base image. Compatible hardware and host drivers still matter.

</details>

### From an environment definition to an image

- [`environment.yml`](environment.yml): the required packages.
- [`Containerfile`](Containerfile): the build recipe, using Dockerfile syntax.

#### The package requirements

Our [`environment.yml`](environment.yml) includes the analysis packages, HTTP data-access libraries, and `ipykernel` for Jupyter:

```yaml
name: atlas-notebook-find-z
channels:
  - conda-forge
  - nodefaults
dependencies:
  - python=3.11.16
  - aiohttp=3.14.3
  - atlasopenmagic=1.10.1
  - awkward=2.13.0
  - ipykernel=6.31.0
  - matplotlib-base=3.11.1
  - numpy=2.4.6
  - pandas=2.3.3
  - requests=2.34.2
  - uproot=5.7.6
  - vector=1.8.1
```

Packages come from **conda-forge**. `ipykernel` runs Python for Jupyter; `requests` and `aiohttp` support remote data access. NERSC supplies the Jupyter server.

#### The build instructions

Our [`Containerfile`](Containerfile):

```dockerfile
FROM docker.io/mambaorg/micromamba:2.9.0

COPY --chmod=644 environment.yml /opt/environment.yml
RUN micromamba install --yes --name base --file /opt/environment.yml && \
    micromamba clean --all --yes

WORKDIR /tmp
CMD ["python"]
```

- **`FROM`:** start from an image containing Micromamba.
- **`COPY`:** add the environment file and make it readable during installation.
- **`RUN`:** install the packages and clean up download caches.

The image contains the software; notebooks, analysis scripts, and data stay outside it.

<details>
<summary>Environment and runtime details</summary>

`--name base` installs into the image’s base environment, whose Python is `/opt/conda/bin/python`. The image inherits the non-root `mambauser` (UID/GID `57439`); `--chmod=644` makes the copied YAML readable even with restrictive checkout permissions.

Keep the base image’s entrypoint: it activates the environment. The image starts in `/tmp`; Section 4 mounts your files and selects the analysis directory after startup.

See the [Micromamba container guide](https://micromamba-docker.readthedocs.io/en/latest/quick_start.html).

</details>

### Build our analysis image

**Podman-HPC** builds and runs containers at NERSC.

**Where to run:** use a **Perlmutter login-node terminal**, or **NERSC Python on a login node**. Do not run these Bash commands inside the ATLAS container kernel. In a terminal, omit `%%bash`; run from the repository directory.

In [ ]:
%%bash
podman-hpc build --file Containerfile --tag localhost/atlas-notebook-find-z:v1 .

- **`--file Containerfile`:** the build recipe.
- **`--tag`:** the image name used throughout this notebook.
- **`.`:** the build context—the current directory.

**Expected result:** the build finishes and tags the image.

<details>
<summary>Build troubleshooting</summary>

If a build fails, inspect the first failing instruction. For `pasta` network-helper failures, see [NERSC’s network options](https://docs.nersc.gov/development/containers/podman-hpc/overview/#network-options-with-the-container); `--network=slirp4netns` is a possible fallback after `build` or `run`.

After changing the environment, rebuild and migrate the image, restart the container kernel, and rerun the checks.

</details>

### Check the environment

First, list just the tutorial image:

In [ ]:
%%bash
podman-hpc images --filter reference=localhost/atlas-notebook-find-z:v1

The listing should include the tutorial image. Now start a temporary container and check that the required packages import.

`--rm` removes the container when it exits; the image remains.

In [ ]:
%%bash
podman-hpc run --rm localhost/atlas-notebook-find-z:v1 python -c '
import atlasopenmagic, awkward, ipykernel, matplotlib, numpy, pandas, uproot, vector
import requests, aiohttp
print("Analysis imports OK")
'

**Expected result:** `Analysis imports OK`, with no import errors. Section 5 will test data access by running the analysis.

### Make the image available to Jupyter

On the build node, use `podman-hpc migrate` to make the image available on shared storage for other nodes. See [NERSC’s image guidance](https://docs.nersc.gov/development/containers/podman-hpc/overview/#building-images).

In [ ]:
%%bash
podman-hpc migrate localhost/atlas-notebook-find-z:v1
podman-hpc images --filter reference=localhost/atlas-notebook-find-z:v1

### ⚑ Checkpoint: Software environment ready

> The image builds successfully, and the dependency checks pass.

Confirm that the image listing contains a migrated copy (`R/O` is `true`) before continuing to Section 4.

<details>
<summary>Further reading</summary>

- [Podman-HPC beginner tutorial](https://docs.nersc.gov/development/containers/podman-hpc/podman-beginner-tutorial/).
- [Using Podman-HPC at NERSC](https://docs.nersc.gov/development/containers/podman-hpc/overview/).
- [Dockerfile reference](https://docs.docker.com/reference/dockerfile/).

</details>

## 4 - Containers on NERSC JupyterHub

### Keep the notebook; change the kernel

The **kernel** executes your notebook’s code. A **kernelspec** (`kernel.json`) tells Jupyter which command starts it.

We’ll generate the kernelspec with `ipykernel`, then prepend the container launch command using `jq`. See [NERSC’s container-kernel guide](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-use-a-container-to-run-a-jupyter-kernel).

**Where to run setup:** use **NERSC Python**, not the ATLAS container kernel, or a **host terminal** with `%%bash` omitted. You need `podman-hpc`, `jq`, the migrated image, and a checkout under your home directory.

**Alternative:** [Shifter-backed kernels](https://docs.nersc.gov/services/jupyter/how-to-guides/#shifter) use the same principle, with different image preparation and launch options.

### 1. Include ipykernel in the container

Our environment already includes `ipykernel`, which lets the container’s Python communicate with Jupyter.

### 2. Generate and configure the kernelspec

The cell below uses the container’s Python to generate `kernel.json`, then uses `jq` to prepend the container launch command to `argv`. The generated Python arguments and `{connection_file}` placeholder are preserved.

`--prefix "$HOME/.local"` installs the kernelspec in your mounted home directory. Rerunning the whole cell replaces this tutorial’s kernelspec with a fresh one before adding the wrapper.

In [ ]:
%%bash
set -euo pipefail

spec="$HOME/.local/share/jupyter/kernels/atlas-notebook-find-z/kernel.json"
image="localhost/atlas-notebook-find-z:v1"

# Generate a fresh kernelspec.
podman-hpc run --rm --jupyter \
    --userns=keep-id:uid=57439,gid=57439 \
    --env HOME \
    "$image" \
    /opt/conda/bin/python -m ipykernel install \
    --prefix "$HOME/.local" \
    --name atlas-notebook-find-z \
    --display-name "ATLAS Find Z (container)"

# Launch the generated Python command inside the container.
jq --arg image "$image" '
    .argv = [
        "podman-hpc", "run", "--rm", "--jupyter",
        "--userns=keep-id:uid=57439,gid=57439",
        "--env", "HOME",
        $image
    ] + .argv
' "$spec" > "$spec.tmp"
mv "$spec.tmp" "$spec"

jq . "$spec"

Jupyter will now launch the container before starting Python. We write the edited JSON to a temporary file, then replace `kernel.json`, because `jq` cannot read and overwrite the same file directly.

<details>
<summary>What do the container options do?</summary>

- `--jupyter` mounts home and `/tmp`; `--env HOME` passes your home path.
- `--userns=keep-id:uid=57439,gid=57439` maps your account to this image’s non-root user so it can access your files. It does not change host file ownership.

The numeric IDs belong to this image, not your NERSC account. See [Podman’s user mapping documentation](https://docs.podman.io/en/latest/markdown/podman-run.1.html#userns-mode).

</details>

### 3. Select the container kernel

After creating or editing the kernelspec, **save the notebook and reload the JupyterLab browser tab**. Then choose **Kernel → Change Kernel → ATLAS Find Z (container)**. If it is still missing, restart your server using the steps below.

<details>
<summary>If the kernel does not appear: restart your Jupyter server</summary>

1. Save all open notebooks; stopping the server ends its kernels and clears in-memory variables.
2. Select **File → Hub Control Panel**.
3. Once the control panel opens, close the old JupyterLab tab.
4. Stop your running server, wait for it to stop, then start a new server with the resources you need.
5. Reopen this notebook and select **ATLAS Find Z (container)**.

This restarts your server, not the shared JupyterHub service. [NERSC’s restart guidance](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-use-a-conda-environment-as-a-python-kernel) also distinguishes this from restarting an existing kernel after changing its launch settings.

</details>

Switching kernels starts a fresh Python session. Run the verification cells below, not the setup cell again.

### 4. Verify the environment

Check the Python interpreter and select the repository directory. The image starts in `/tmp`.

> **Before running the next cell:** set `repo` to the location where you cloned this repository. The example assumes `~/notebook-laptop-to-nersc`. If you cloned into another directory under your home directory, update the path to match.

In [ ]:
import os
import sys
from pathlib import Path

print("Python:", sys.executable)
assert sys.executable == "/opt/conda/bin/python"

# EDIT this path if you cloned the repository elsewhere under your home directory.
repo = Path.home() / "notebook-laptop-to-nersc"
assert (repo / "utils/find_the_z.py").is_file(), f"Check the repository path: {repo}"
os.chdir(repo)
print("Analysis directory:", Path.cwd())

In [ ]:
import atlasopenmagic, awkward, ipykernel, matplotlib, numpy, pandas, uproot, vector
import requests, aiohttp
from utils.find_the_z import process_data, plot_result

print("Analysis imports OK")

### ⚑ Checkpoint: Jupyter connection ready

> The selected kernel runs Python inside the container and imports the analysis packages.

<details>
<summary>Troubleshooting and further reading</summary>

- **Image unavailable:** check the image name and migration under your account.
- **Permission denied:** check the user mapping and mounted paths; do not change ownership of your home directory.
- **Kernel disconnected:** inspect the kernelspec and Jupyter server error output.
- **Missing dependencies:** update the environment, rebuild and migrate the image, then restart the kernel.
- **Local module missing:** check the repository path and current directory.

References: [NERSC kernels](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-use-a-container-to-run-a-jupyter-kernel), [Jupyter kernelspecs](https://jupyter-client.readthedocs.io/en/stable/kernels.html#kernel-specs), and the [jq manual](https://jqlang.org/manual/).

</details>

## 5 - Running the Analysis

Keep **ATLAS Find Z (container)** selected.

### 1. Access the example data

Select the tutorial’s data release and `2muons` skim (events with at least two muons, each with transverse momentum of at least 10 GeV). We’ll process one file.

In [ ]:
import atlasopenmagic as atom

# Select the ATLAS Open Data release used for this tutorial.
atom.set_release('2025e-13tev-beta')
skim = '2muons'

files_list = atom.get_urls('data', skim, protocol='https', cache=True)
print(f"Available files: {len(files_list)}")

<details>
<summary>Optional: check a single event</summary>

After the data-access cell, run this in the container kernel to check the required ROOT branches before processing.

```python
import uproot

assert len(files_list) > 0, "No data files were returned; check the data-access step."
with uproot.open(f"{files_list[0]}:analysis") as tree:
    sample = tree.arrays(
        ["lep_pt", "lep_eta", "lep_phi", "lep_e"],
        entry_start=0,
        entry_stop=1,
        library="ak",
    )

assert len(sample) == 1, "The selected file contains no events."
print("Data-access check passed: read one event and all four required branches.")
```

[Uproot reading guide](https://uproot.readthedocs.io/en/stable/basic.html).

</details>

### 2. Run the analysis

The [analysis functions](utils/find_the_z.py) read the muon measurements and calculate the pair’s invariant mass. For this demonstration, use the first file; the function processes half its events.

In [ ]:
from utils.find_the_z import process_data, plot_result

assert len(files_list) > 0, "No data files were returned; check the data-access step."
demo_files = files_list[:1]
mass_list = process_data(demo_files)

### 3. Plot and interpret

Look for a peak near **91 GeV**, the Z boson mass. The vertical axis shows events per mass bin on a logarithmic scale.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plot_result(mass_list)
plt.show()

### ⚑ Checkpoint: Analysis verified

> Data access works, and the plot shows a peak near the Z boson mass.


## 6 - Summary and Further Exploration

### What we accomplished

We ran the ATLAS analysis through a container-backed Jupyter kernel. You’ve learned to:

- **Build a container image** containing your analysis software.
- **Configure Jupyter to use it**, while retaining your notebook interface.

### Where to go next

**Batch jobs:** reuse the environment from a Slurm job, launching an analysis script instead of a Jupyter kernel.

<details>
<summary>Try it later</summary>

Write a small driver that calls `process_data` and `plot_result`, saves the figure, and runs inside the container from a batch script. This suits longer or repeated analyses without interactive input.

</details>

**GPUs:** extend the workflow to a GPU-enabled notebook.

<details>
<summary>Try it later</summary>

1. Build an image with a compatible GPU-enabled library such as CuPy or PyTorch, plus `ipykernel`.
2. Start a NERSC Jupyter session with GPU resources.
3. Add `--gpu` alongside `--jupyter` in the kernel launch command.
4. Check GPU detection and run a small calculation.

See [NERSC’s GPU container guidance](https://docs.nersc.gov/development/containers/podman-hpc/overview/#using-nvidia-gpus-in-podman-hpc).

</details>

**Sharing:** publish the image so collaborators can reuse the environment.

<details>
<summary>Try it later</summary>

Push a versioned image to NERSC’s private, project-based registry. Collaborators with access can pull it instead of rebuilding. Follow the [NERSC registry guide](https://docs.nersc.gov/development/containers/registry/); you may need to request access.

</details>

**Automation:** build, test, and publish images with GitHub Actions.

<details>
<summary>Try it later</summary>

Create a workflow that builds the image, runs checks you define, and publishes to GitHub Container Registry (`ghcr.io`). Tag builds with release or commit identifiers, then pull the image at NERSC.

[GitHub’s publishing guide](https://docs.github.com/en/actions/tutorials/publish-packages/publish-docker-images) covers build and publication; add your own test steps.

</details>

Bring your own scientific notebook next: **package its environment, connect it to Jupyter, and continue your science at NERSC.**

<details>
<summary>Useful resources</summary>

- [Podman-HPC tutorial](https://docs.nersc.gov/development/containers/podman-hpc/podman-beginner-tutorial/).
- [Container-backed Jupyter kernels](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-use-a-container-to-run-a-jupyter-kernel).
- [Micromamba container guide](https://micromamba-docker.readthedocs.io/en/latest/quick_start.html).

</details>